<a href="https://colab.research.google.com/github/charlottesscott/Breast_Screening_Uptake_Prediction/blob/main/LA_coverage_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Import Libaries**

In [2]:
import io
import pandas as pd

**Upload the CSV**

In [3]:
from google.colab import files

uploaded_files = files.upload()

Saving kc63_coverage_24-25.csv to kc63_coverage_24-25.csv


**Read the CSV file**

In [4]:
df = pd.read_csv("kc63_coverage_24-25.csv")

**Check CSV**

In [5]:
df.shape
df.head()

,CollectionYearRange,Parent_OrgONSCode,Parent_Org_Name,Org_ONSCode,Org_Name,Org_Type,Age_Band,Women_resident,Women_ineligible,Women_eligible,Women_never_screened,Percent_never_screened,Women_screened_less3yrs,Coverage
0,2014-15,E92000001,England,E92000001,England,National,45-49,2021433,573,2020860,1768799,87.52704294,251042,12.422533
1,2014-15,E92000001,England,E92000001,England,National,50,409486,352,409134,226553,55.37378952,178826,43.708418
2,2014-15,E92000001,England,E92000001,England,National,51-52,793873,862,793011,276121,34.81931524,502582,63.376422
3,2014-15,E92000001,England,E92000001,England,National,53-54,749463,991,748472,153854,20.55574557,551576,73.693605
4,2014-15,E92000001,England,E92000001,England,National,53<71,5750911,9928,5740983,670738,11.68333019,4329310,75.410605


**Keep LA records only**

In [7]:
la_coverage_data = df.loc[
    df["Org_Type"]
    .astype(str)
    .str.strip()
    .str.upper()
    .eq("LA")
].copy()

In [8]:
print(la_coverage_data["Org_Type"].value_counts())

Org_Type
LA    1647
Name: count, dtype: int64


**Keep the columns need only**

In [9]:
la_coverage_data = la_coverage_data[
    [
        "Org_ONSCode",
        "Org_Name",
        "Org_Type",
        "CollectionYearRange",
        "Coverage"
    ]
].copy()

**Clean the columns**

In [10]:
text_columns = [
    "Org_ONSCode",
    "Org_Name",
    "Org_Type",
    "CollectionYearRange"
]

for column in text_columns:
    la_coverage_data[column] = (
        la_coverage_data[column]
        .astype("string")
        .str.strip()
    )

la_coverage_data["Coverage"] = pd.to_numeric(
    la_coverage_data["Coverage"],
    errors="coerce"
)

**Create the coverage-year column names**

In [11]:
la_coverage_data["Coverage_Year"] = (
    "Coverage_"
    + la_coverage_data["CollectionYearRange"]
)

In [14]:
la_coverage_data.shape
la_coverage_data.head()

,Org_ONSCode,Org_Name,Org_Type,CollectionYearRange,Coverage,Coverage_Year
119,E06000001,Hartlepool,LA,2023-24,68.500665,Coverage_2023-24
120,E06000002,Middlesbrough,LA,2023-24,68.087713,Coverage_2023-24
202,E06000003,Redcar and Cleveland,LA,2023-24,72.759524,Coverage_2023-24
203,E06000004,Stockton-on-Tees,LA,2023-24,72.639748,Coverage_2023-24
204,E06000005,Darlington,LA,2023-24,72.984525,Coverage_2023-24


**Reshape the Data**

In [19]:
# Sort the data so the most recent organisation name appears last
la_coverage_data = la_coverage_data.sort_values(
    "CollectionYearRange"
)

# Create a lookup containing the most recent name for each ONS code
organisation_lookup = (
    la_coverage_data[
        ["Org_ONSCode", "Org_Name", "Org_Type"]
    ]
    .drop_duplicates(
        subset=["Org_ONSCode"],
        keep="last"
    )
)

# Create a column heading for each year
la_coverage_data["Coverage_Year"] = (
    "Coverage_"
    + la_coverage_data["CollectionYearRange"]
)

# Pivot using the ONS code only
la_coverage_wide = (
    la_coverage_data.pivot_table(
        index="Org_ONSCode",
        columns="Coverage_Year",
        values="Coverage",
        aggfunc="first"
    )
    .reset_index()
)

# Remove the extra pivot heading
la_coverage_wide.columns.name = None

# Add the most recent organisation name and type
la_coverage_wide = la_coverage_wide.merge(
    organisation_lookup,
    on="Org_ONSCode",
    how="left"
)

**Put the columns in year order**

In [20]:
identifier_columns = [
    "Org_ONSCode",
    "Org_Name",
    "Org_Type"
]

coverage_columns = sorted(
    column
    for column in la_coverage_wide.columns
    if column.startswith("Coverage_")
)

la_coverage_wide = (
    la_coverage_wide[
        identifier_columns + coverage_columns
    ]
    .sort_values(["Org_Name", "Org_ONSCode"])
    .reset_index(drop=True)
)

**Sort the local authorities**

In [21]:
la_coverage_wide = (
    la_coverage_wide
    .sort_values(["Org_Name", "Org_ONSCode"])
    .reset_index(drop=True)
)

**View the final table**

In [22]:
display(la_coverage_wide.head())

,Org_ONSCode,Org_Name,Org_Type,Coverage_2014-15,Coverage_2015-16,Coverage_2016-17,Coverage_2017-18,Coverage_2018-19,Coverage_2019-20,Coverage_2020-21,Coverage_2021-22,Coverage_2022-23,Coverage_2023-24,Coverage_2024-25
0,E09000002,Barking and Dagenham,LA,64.326780,66.585250,67.746341,66.967384,65.830587,66.364552,54.516350,59.930104,61.006289,64.947043,64.892882
1,E09000003,Barnet,LA,70.498073,69.911226,68.897813,69.528927,68.855425,68.270282,57.127619,61.901013,58.458507,63.452957,67.328100
2,E08000016,Barnsley,LA,80.840217,79.959522,79.807241,80.145382,79.919778,76.097628,55.496043,67.942311,69.929815,73.177218,68.671679
3,E06000022,Bath and North East Somerset,LA,75.569369,76.107315,75.824450,75.350084,74.736687,75.377687,72.248210,69.991062,70.822256,70.946816,73.245375
4,E06000055,Bedford,LA,72.116780,73.135175,73.103562,73.705440,74.325556,73.812271,58.402885,62.676872,61.157962,67.429315,67.743167


**Save Output**

In [23]:
output_file_name = "la_coverage_by_year.csv"

la_coverage_wide.to_csv(
    output_file_name,
    index=False
)

files.download(output_file_name)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>